[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/Demo_29_ActorCritic_ContinuousControl.ipynb)

# Demo 29: Actor–Critic: Continuous Balance Control

**挿入位置：Actor–Critic・PPO・SACの導入**　｜　**実行環境：Google Colab**

架空の**1自由度バランス制御**（自転車の傾きに見立てた簡略モデル）を用い、Actorが連続的なトルクを選び、Criticが状態価値を評価する仕組みを可視化します。MuJoCoの厳密な自転車モデルではありません。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
class BalanceToy:
    def __init__(self,limit=80):self.limit=limit
    def reset(self,rng,initial=None):
        self.theta=float(rng.uniform(-.25,.25) if initial is None else initial);self.omega=0.0;self.t=0
        return np.array([self.theta,self.omega],dtype=np.float32)
    def step(self,action,rng):
        u=float(np.clip(action,-2.,2.));dt=.06
        self.omega+=dt*(3*self.theta + 3*u - .5*self.omega + rng.normal(0,.05))
        self.theta+=dt*self.omega;self.t+=1
        failed=abs(self.theta)>.7;done=failed or self.t>=self.limit
        reward=1.0 - 3*self.theta**2 - .3*self.omega**2 - .03*u*u - (8.0 if failed else 0.)
        return np.array([self.theta,self.omega],dtype=np.float32),float(reward),done,failed

class Actor(nn.Module):
    def __init__(self,width=32):
        super().__init__();self.net=nn.Sequential(nn.Linear(2,width),nn.Tanh(),nn.Linear(width,width),nn.Tanh(),nn.Linear(width,1))
    def forward(self,s):return 1.8*torch.tanh(self.net(s)).squeeze(-1)

class Critic(nn.Module):
    def __init__(self,width=32):
        super().__init__();self.net=nn.Sequential(nn.Linear(2,width),nn.Tanh(),nn.Linear(width,1))
    def forward(self,s):return self.net(s).squeeze(-1)

def train_actor_critic(episodes=220,seed=0,width=32):
    torch.manual_seed(seed);rng=np.random.default_rng(seed)
    actor=Actor(width);critic=Critic(width)
    optA=torch.optim.Adam(actor.parameters(),lr=.002);optC=torch.optim.Adam(critic.parameters(),lr=.004)
    env=BalanceToy();returns=[];fallen=[];initial_state={k:v.clone() for k,v in actor.state_dict().items()}
    for ep in range(episodes):
        state=env.reset(rng);epret=0;failed=False
        states=[];nextstates=[];actions=[];rewards=[];dones=[]
        for t in range(env.limit):
            # Collect transitions without retaining a large computation graph.
            with torch.no_grad():
                st=torch.tensor(state,dtype=torch.float32)
                mean=actor(st);action=torch.distributions.Normal(mean,torch.tensor(.4)).sample().clamp(-2,2)
            ns,r,done,failed=env.step(float(action),rng)
            states.append(state.copy());nextstates.append(ns.copy());actions.append(float(action));rewards.append(r);dones.append(done)
            state=ns;epret+=r
            if done:break
        states_t=torch.tensor(np.array(states),dtype=torch.float32)
        next_t=torch.tensor(np.array(nextstates),dtype=torch.float32)
        actions_t=torch.tensor(actions,dtype=torch.float32)
        rewards_t=torch.tensor(rewards,dtype=torch.float32)
        dones_t=torch.tensor(dones,dtype=torch.float32)
        with torch.no_grad():
            target=rewards_t+.98*(1-dones_t)*critic(next_t)
        delta=target-critic(states_t)
        # One batched update per episode: semi-gradient TD critic + advantage-weighted actor.
        lossC=.5*delta.pow(2).mean()
        optC.zero_grad();lossC.backward();nn.utils.clip_grad_norm_(critic.parameters(),5);optC.step()
        with torch.no_grad():
            adv=(target-critic(states_t)).clamp(-5,5)
        mu=actor(states_t)
        dist=torch.distributions.Normal(mu,torch.tensor(.4))
        lossA=-(dist.log_prob(actions_t)*adv).mean()
        optA.zero_grad();lossA.backward();nn.utils.clip_grad_norm_(actor.parameters(),5);optA.step()
        returns.append(epret);fallen.append(int(failed))
    first=Actor(width);first.load_state_dict(initial_state)
    return first,actor,critic,np.array(returns),np.array(fallen)

def trace(actor,seed=77,theta=.23):
    env=BalanceToy();rng=np.random.default_rng(seed);state=env.reset(rng,initial=theta)
    states=[state.copy()];acts=[];rewards=[]
    for t in range(env.limit):
        with torch.no_grad():a=float(actor(torch.tensor(state)).item())
        state,r,done,failed=env.step(a,rng);acts.append(a);rewards.append(r);states.append(state.copy())
        if done:break
    return np.array(states),np.array(acts),np.array(rewards)

In [ ]:
epochs=widgets.IntSlider(value=220,min=70,max=500,step=30,description='Episodes',continuous_update=False)
seed=widgets.IntSlider(value=7,min=0,max=100,description='Seed')
angle=widgets.FloatSlider(value=.23,min=-.4,max=.4,step=.01,description='Initial tilt',continuous_update=False)
button=widgets.Button(description='Train Actor–Critic',button_style='primary');out=widgets.Output()
def run(_):
    with out:
        clear_output(wait=True);print('Training simplified continuous-control Actor–Critic...')
        first,actor,critic,returns,fallen=train_actor_critic(epochs.value,seed.value)
        f,fa,fr=trace(first,theta=angle.value);a,aa,ar=trace(actor,theta=angle.value)
        clear_output(wait=True)
        fig,ax=plt.subplots(1,3,figsize=(14.5,3.8))
        ax[0].plot(returns,alpha=.3);w=min(25,len(returns)//2);ax[0].plot(np.arange(w,len(returns)+1),np.convolve(returns,np.ones(w)/w,'valid'),label='moving mean');ax[0].set(title='Training returns',xlabel='Episode');ax[0].legend()
        ax[1].plot(f[:,0],label='Before');ax[1].plot(a[:,0],label='After');ax[1].axhline(.7,color='r',ls=':');ax[1].axhline(-.7,color='r',ls=':');ax[1].set(title='Tilt θ (single test)',xlabel='Step',ylabel='Angle');ax[1].legend()
        ax[2].plot(fa,label='Actor before');ax[2].plot(aa,label='Actor after');ax[2].set(title='Continuous steering torque',xlabel='Step',ylabel='Torque');ax[2].legend()
        for x in ax:x.grid(alpha=.2)
        plt.tight_layout();plt.show()
        xs=torch.tensor(np.c_[np.linspace(-.4,.4,100),np.zeros(100)],dtype=torch.float32)
        with torch.no_grad():u=actor(xs).numpy();v=critic(xs).numpy()
        plt.figure(figsize=(8,3));plt.plot(xs[:,0],u,label='Actor mean action');plt.plot(xs[:,0],v,label='Critic V(s)');plt.axhline(0,color='k',lw=.5);plt.xlabel('Tilt θ (speed=0)');plt.title('Two learned components');plt.legend();plt.grid(alpha=.2);plt.show()
        print('Test duration: Before=',len(fa),'After=',len(aa),'steps; training failure fraction (last 30)=',round(fallen[-30:].mean(),2))
button.on_click(run)
display(widgets.VBox([widgets.HBox([epochs,seed,angle]),button,out]))
print('Click Train to see before/after and the learned policy/value functions.')

### 学習のポイント
- **Actor**：$\pi_\theta(a\mid s)$を表現する連続値の確率分布（ここでは固定標準偏差を持つ正規分布）からトルクを選びます。
- **Critic**：状態価値 $V(s)$ を推定し、$\delta_t=r_t+\gamma V(s_{t+1})-V(s_t)$ で更新します。
- ActorはCriticのTD誤差を方策勾配の重みとして使用します。
- これは**簡略化した学習用Actor–Critic**であり、PPO/SACの実装そのものではありません。トルク上限と転倒条件を設けています。

## Try it yourself / 実験課題
- 学習前と学習後の角度の時系列を比較する。
- Actorの操作とCriticの値が別の役割であることを説明する。
- 初期傾きを大きくして、学習済み方策の限界を確かめる。